# Comparação de mapas: intensidade absoluta, MEBin e contraste local

Execute as células iniciais com o ZIP atualizado. A análise exploratória reutiliza os mapas históricos; a avaliação independente requer mapas de normais reservadas fora do treino. Nenhuma rotina altera o checkpoint ou os rótulos históricos.

MEBin: variante com faixa min/max por imagem, sem erosão por padrão para preservar defeitos pequenos; não é reprodução exata do paper, que determina a faixa pelo conjunto. Contraste: mediana em duas escalas com escala robusta dos resíduos locais; não é uma probabilidade.

Referências: [MEBin](https://github.com/HUST-SLOW/MEBin), [calibração conformal](https://arxiv.org/abs/2104.08279).


In [2]:
import importlib.metadata
import os
import subprocess
import sys

REQUIRED_PACKAGES = {
    "anomalib": "2.6.0",
    "ultralytics-opencv-headless": "8.4.123",
}

def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None

needs_installation = any(
    installed_version(package) != version
    for package, version in REQUIRED_PACKAGES.items()
)

if needs_installation:
    print("Instalando o ambiente fixo do PatchCore...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade-strategy",
        "only-if-needed",
        "anomalib==2.6.0",
        "ultralytics-opencv-headless==8.4.123",
    ])
    print("Instalacao concluida. Reinicie o kernel antes de continuar.")
    raise SystemExit("Reinicie o kernel manualmente e execute esta celula novamente.")

print("Ambiente configurado. Continue para a proxima celula.")


Instalando o ambiente fixo do PatchCore...


: 

: 

: 

## 2. Disponibilizar o código ao kernel

No Colab, monte o Drive uma vez. A célula procura a pasta `insulator_anomaly` ou o ZIP de código em Meu Drive e em `Meu Drive/Colab_Notebooks`. Para outro local, preencha `CODE_PATH` (pasta que contém o pacote) ou `CODE_ZIP_PATH`.


In [1]:
from pathlib import Path
import sys
import importlib
import tempfile
import zipfile

CODE_PATH = ""
CODE_ZIP_PATH = ""
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")

if CODE_PATH and CODE_ZIP_PATH:
    raise ValueError("Preencha apenas CODE_PATH ou CODE_ZIP_PATH.")
code_roots = [Path(CODE_PATH)] if CODE_PATH else [Path.cwd()]
if IN_COLAB and not CODE_PATH:
    code_roots += [Path("/content/drive/MyDrive"), Path("/content/drive/MyDrive/Colab_Notebooks")]
code_root = None if CODE_ZIP_PATH else next(
    (root for root in code_roots if (root / "insulator_anomaly" / "__init__.py").is_file() and (root / "insulator_anomaly" / "crop_experiment.py").is_file() and (root / "insulator_anomaly" / "map_postprocessing.py").is_file()), None)
if code_root is None:
    code_archive = Path(CODE_ZIP_PATH) if CODE_ZIP_PATH else next(
        (root / "insulator_anomaly.zip" for root in code_roots if (root / "insulator_anomaly.zip").is_file()), None)
    if code_archive is None or not code_archive.is_file():
        raise FileNotFoundError("Disponibilize insulator_anomaly/ ou insulator_anomaly.zip ao kernel.")
    code_root = Path(tempfile.mkdtemp(prefix="insulator_code_")).resolve()
    with zipfile.ZipFile(code_archive) as bundle:
        if not {"insulator_anomaly/crop_experiment.py", "insulator_anomaly/map_postprocessing.py"}.issubset(bundle.namelist()):
            raise FileNotFoundError("ZIP antigo: envie o insulator_anomaly.zip atualizado, contendo crop_experiment.py.")
        for member in bundle.infolist():
            if not (code_root / member.filename).resolve().is_relative_to(code_root):
                raise ValueError(f"Caminho inválido no ZIP de código: {member.filename}")
        bundle.extractall(code_root)
    if not (code_root / "insulator_anomaly" / "__init__.py").is_file():
        raise FileNotFoundError("O ZIP de código não contém o pacote insulator_anomaly.")
for module_name in list(sys.modules):
    if module_name == "insulator_anomaly" or module_name.startswith("insulator_anomaly."):
        del sys.modules[module_name]
importlib.invalidate_caches()
sys.path.insert(0, str(code_root))
print("Código:", code_root)

import insulator_anomaly.crop_experiment as crop_module
print("Experimento carregado:", crop_module.__file__)


Mounted at /content/drive
Código: /tmp/insulator_code_u4hitfx0
Experimento carregado: /tmp/insulator_code_u4hitfx0/insulator_anomaly/crop_experiment.py


## 3. Configurar o projeto

No Colab, o Python executa em outra máquina e não acessa `C:/...`. Deixe os campos vazios para procurar os dados junto ao notebook ou no Drive; preencha um deles para outra localização.

`ProjectConfig` centraliza parâmetros e caminhos. Construir a configuração não exige as imagens brutas, permitindo avaliar apenas os dados processados. `background_modes` pode selecionar os modos que deseja executar.


In [2]:
from insulator_anomaly.config import ProjectConfig
from insulator_anomaly.paths import resolve_project

PROJECT_PATH = ""  # Ex.: "/content/drive/MyDrive/Colab_Notebooks/anomalib_glass"
ZIP_PATH = ""      # Ex.: "/content/drive/MyDrive/anomalib_glass.zip"
project_root, persistent_root = resolve_project(PROJECT_PATH, ZIP_PATH, colab=IN_COLAB)
import torch

ACCELERATOR = "gpu"  # Use "cpu" apenas para executar sem GPU.
if ACCELERATOR == "gpu":
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA indisponivel. Instale PyTorch com CUDA no ambiente do kernel "
            "e confira o driver NVIDIA. Consulte README.md."
        )
    probe = torch.ones(1, device="cuda") * 2
    assert probe.item() == 2
    del probe
    print("GPU:", torch.cuda.get_device_name(0), "| CUDA:", torch.version.cuda)
config = ProjectConfig(project_root, persistent_root, accelerator=ACCELERATOR, num_workers=0)
print("Dados:", config.project_root)
print("Saídas persistentes:", config.persistent_results_root)
print("Modos:", config.background_modes)


Dados: /content/drive/MyDrive/Colab_Notebooks/anomalib_glass
Saídas persistentes: /content/drive/MyDrive/Colab_Notebooks/anomalib_glass/patchcore_insulators/results
Modos: ('original', 'blur', 'gray')


## A. Mapas históricos: exploração sem nova inferência

Requer `heatmap_metrics.csv` e mapas `.npy` gerados pela seção 9 do notebook principal. Usa o limiar pixel histórico; o contraste usa um piso por imagem. Não estima desempenho de todo o teste, não gera decisões calibradas e não altera `pred_label`. As máscaras são candidatos, não confirmação do defeito. Figuras ficam salvas no destino impresso.

**Saída desta seção:** `results/<modo>/map_comparison_<data>/`, com figuras, máscaras e `exploratory_metrics.csv`. Para comparar visualmente os heatmaps históricos, encerre aqui. Esta seção não gera `map_manifest.csv`.


In [ ]:
from datetime import datetime
import pandas as pd
from insulator_anomaly.map_postprocessing import MapSettings, explore_saved_maps

RUN_EXPLORATION = True  # Mude para True para comparar os mapas históricos.
MAP_MODE = "gray"
if RUN_EXPLORATION:
    csv_path = config.persistent_results_root / MAP_MODE / "heatmap_analysis" / "heatmap_metrics.csv"
    saved = pd.read_csv(csv_path)
    # Se os arquivos foram copiados de uma sessão temporária, procura pelo nome na pasta persistente.
    saved["map_path"] = saved.map_path.map(
        lambda p: str(Path(p) if Path(p).is_file() else csv_path.parent / Path(p).name)
    )
    output = config.persistent_results_root / MAP_MODE / (
        "map_comparison_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    details = explore_saved_maps(saved, output, MapSettings(), show_first=8)
    display(details[["image_path", "method", "score", "area_fraction", "components", "pixel_threshold"]])
    print("Figuras e máscaras:", output)


gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comparados.
gray: 1/1 mapas comp

,image_path,method,score,area_fraction,components,pixel_threshold
0,/content/drive/MyDrive/Colab_Notebooks/anomali...,absolute,0.391591,0.000000,0,0.512250
1,/content/drive/MyDrive/Colab_Notebooks/anomali...,mebin,0.391591,0.059708,1,0.243965
2,/content/drive/MyDrive/Colab_Notebooks/anomali...,contrast,0.881315,0.000000,0,3.000000
3,/content/drive/MyDrive/Colab_Notebooks/anomali...,absolute,0.381955,0.000000,0,0.512250
4,/content/drive/MyDrive/Colab_Notebooks/anomali...,mebin,0.381955,0.062851,1,0.250082
...,...,...,...,...,...,...
421,/content/drive/MyDrive/Colab_Notebooks/anomali...,mebin,0.495324,0.994934,1,0.006232
422,/content/drive/MyDrive/Colab_Notebooks/anomali...,contrast,0.734921,0.000000,0,3.000000
423,/content/drive/MyDrive/Colab_Notebooks/anomali...,absolute,0.354311,0.000000,0,0.512250
424,/content/drive/MyDrive/Colab_Notebooks/anomali...,mebin,0.354311,0.298340,1,0.177090


Figuras e máscaras: /content/drive/MyDrive/Colab_Notebooks/anomalib_glass/patchcore_insulators/results/gray/map_comparison_20261027_134922_290500


## B. Opcional: avaliação independente, usando apenas normais para calibrar

**Esta seção não é continuação da seção A.** A entrada vem de outro notebook: `Crop_Comparison.ipynb`, etapa 3. Use em `MAP_EXPERIMENT_ROOT` o mesmo `EXPERIMENT_ROOT` configurado nele (por padrão, `experiments/crop_vs_segments_v1`). Não use a pasta `map_comparison_...`: ela contém apenas a exploração histórica, sem os dados de validação necessários. Os resultados desta avaliação ficam em uma subpasta `map_evaluation_<data>` do experimento.

`run_crop_experiment` agora salva `map_manifest.csv` e os mapas de validação/teste. Se já treinou com uma versão anterior, execute novamente a etapa 3 de Crop_Comparison: reutiliza checkpoints compatíveis e refaz a inferência para exportar os mapas.

O manifesto requer mode, split, source_id, group_id, y_true, image_path e map_path. Pode fornecer defect_path com máscara binária real alinhada ao mapa; máscara YOLO do objeto não é anotação de defeito.

Divide os grupos normais de `val` em referência (fixar escalas) e calibração (decisão conformal). Também aceita splits explícitos reference/calibration/test. O usuário deve garantir que essas normais não participaram do treino. Não reserve referências a partir do teste histórico.

Calibração e alarme usam o máximo por grupo. Havendo várias fotos por grupo, o alarme do grupo é propagado às suas fotos; avalie principalmente unit=group. Sem grupos informados, cada imagem é um grupo e não se exclui vazamento visual. A interpretação conformal requer grupos normais independentes e permutáveis com os futuros grupos normais; não há garantia por pixel.

O score absolute é o top 1% do mapa, não o 
 reponderado do checkpoint. MEBin usa a intensidade original nos candidatos (zero fora deles); contraste usa o top 1% da resposta local. Essas agregações são hipóteses experimentais. Máscara visível pode existir sem alarme. Com poucos grupos normais, a resolução do p-valor pode impedir alarmes; o código avisa. AP/AUROC não escolhem o limiar. Não use teste para ajustar parâmetros e depois alegar avaliação independente.


In [ ]:
from datetime import datetime
from insulator_anomaly.map_postprocessing import MapSettings, evaluate_map_methods
import pandas as pd

RUN_MAP_EVALUATION = False  # Opcional: True após a etapa 3 do Crop_Comparison.
# Mesmo EXPERIMENT_ROOT do Crop_Comparison; não é a saída da seção A.
MAP_EXPERIMENT_ROOT = config.project_root / "experiments" / "crop_vs_segments_v1"
if RUN_MAP_EVALUATION:
    map_manifest = pd.read_csv(MAP_EXPERIMENT_ROOT / "map_manifest.csv")
    map_output = MAP_EXPERIMENT_ROOT / (
        "map_evaluation_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    map_results, map_details = evaluate_map_methods(
        map_manifest, map_output, settings=MapSettings(alpha=0.05), show_first=8,
    )
    display(map_results)
    print("CSVs, curvas PR, figuras e máscaras:", map_output)
